## Tymoteusz Bultrowicz - homework no. 10 - optimizng Fashion MNIST

Notebook przystosowany do odtworzenia lokalnie z środowiskiem uv. \
Środowisko można odtworzyć z załączonego pyproject.toml za pomocą komend: \
**uv venv** \
**uv sync**

Zakłada, że folder roboczy zawiera pobrane z kaggle pliki: \
**fashion-mnist_train.csv** \
**fashion-mnist_test.csv**

In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

I0000 00:00:1780391131.385015   52002 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780391131.385397   52002 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780391131.419778   52002 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780391132.488095   52002 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_E

### Przygotowanie danych - analogicznie do skryptu z laboratorium

In [2]:
train = pd.read_csv('fashion-mnist_train.csv')
test = pd.read_csv('fashion-mnist_test.csv')

X_train = train.drop('label', axis=1).values
X_test = test.drop('label', axis=1).values

y_train = train['label'].values
y_test = test['label'].values

In [3]:
num_classes = np.unique(y_train).shape[0]
# Dataset normalization
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train= X_train/255.
X_test = X_test/255.
# One-hot encoding
y_train_h = np.zeros((np.size(y_train,0) , num_classes))
y_test_h = np.zeros((np.size(y_test,0) , num_classes))
for i in range(np.size(y_train , 0)):
  y_train_h[i , y_train[i]] = 1

for i in range(np.size(y_test , 0)):
  y_test_h[i , y_test[i]] = 1

### Testowanie różnych parametrów modeli

Korzystam z narzędzia optuna. Testuję podane na początku funkcji optimize przedziały hiperparametrów.

In [ ]:
import tensorflow as tf
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import train_test_split

def objective(trial, x_train, y_train, x_val, y_val):
    num_conv_layers = trial.suggest_int('num_conv_layers', 1, 3)
    activation = trial.suggest_categorical('activation', ['relu', 'elu', 'selu'])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    dense_units = trial.suggest_int('dense_units', 32, 128, step=32)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input(shape=(28,28,1)))
    
    # Dodanie warstw konwolucyjnych wyciągających cechy z obrazu.
    for i in range(num_conv_layers):
        filters = trial.suggest_categorical(f'filters_l{i}', [16, 32, 64])
        model.add(tf.keras.layers.Conv2D(
            filters=filters, 
            kernel_size=(3, 3), 
            activation=activation, 
            padding='same'
        ))
        model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
    
    # Warstywa spłaszczająca i gęsta do klasyfikacji.
    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(units=dense_units, activation=activation))
    model.add(tf.keras.layers.Dropout(rate=dropout_rate))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    
    model.compile( # Tworzenie modelu, kompilacja z optymalizatorem Adam.
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    pruning_callback = TFKerasPruningCallback(trial, 'val_accuracy')
    
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=10, # Ilość epok, mocno ograniczona ze względu na czas trwania optymalizacji
        batch_size=64,
        callbacks=[pruning_callback],
        verbose=0  # Wyłączenie logów
    )
    
    val_accuracy = max(history.history['val_accuracy'])
    return val_accuracy


study = optuna.create_study(
    direction="maximize", 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

# Uruchomienie optymalizacji
X_train = X_train.reshape(-1, 28, 28, 1)  
X_test = X_test.reshape(-1, 28, 28, 1)
X_train, X_val, y_train_h, y_val = train_test_split(X_train, y_train_h, test_size=0.2, random_state=42)
study.optimize(lambda trial: objective(trial, X_train, y_train_h, X_val, y_val), n_trials=30, show_progress_bar=True)

# Wyniki końcowe
print("\nNajlepsze parametry:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
    
print(f"\nNajlepsza dokładność walidacyjna: {study.best_value:.4f}")

/home/tymmeek/Desktop/2yINF/MachineLearning/labs/lab11/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/tymmeek/.local/share/uv/python/cpython-3.13.12-linux-x86_64-gnu/lib/python3.13/importlib/__init__.py:88: FutureWarning: `optuna.integration.tfkeras` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.tfkeras` instead.
  return _bootstrap._gcd_import(name[level:], package, level)
[I 2026-06-02 11:05:35,017] A new study created in memory with name: no-name-3f572b2e-f380-463f-aaef-352182efdeab
Best trial: 0. Best value: 0.904917:   3%|▎         | 1/30 [00:43<21:06, 43.67s/it]

[I 2026-06-02 11:06:18,828] Trial 0 finished with value: 0.9049166440963745 and parameters: {'num_conv_layers': 1, 'activation': 'relu', 'dropout_rate': 0.2319848984172977, 'dense_units': 96, 'learning_rate': 0.007368800709739502, 'filters_l0': 32}. Best is trial 0 with value: 0.9049166440963745.


Best trial: 1. Best value: 0.910833:   7%|▋         | 2/30 [02:03<30:26, 65.22s/it]

[I 2026-06-02 11:07:39,137] Trial 1 finished with value: 0.9108333587646484 and parameters: {'num_conv_layers': 2, 'activation': 'selu', 'dropout_rate': 0.4213481778775966, 'dense_units': 128, 'learning_rate': 0.0009073959677901548, 'filters_l0': 64, 'filters_l1': 32}. Best is trial 1 with value: 0.9108333587646484.


Best trial: 1. Best value: 0.910833:  10%|█         | 3/30 [02:59<27:24, 60.91s/it]

[I 2026-06-02 11:08:34,921] Trial 2 finished with value: 0.909250020980835 and parameters: {'num_conv_layers': 2, 'activation': 'relu', 'dropout_rate': 0.30234326172041187, 'dense_units': 64, 'learning_rate': 0.0003159185105802248, 'filters_l0': 32, 'filters_l1': 64}. Best is trial 1 with value: 0.9108333587646484.


... (50 outputs left out)

Best trial: 24. Best value: 0.919833:  93%|█████████▎| 28/30 [-36:03<02:29, 74.65s/it]

[I 2026-06-02 10:29:31,545] Trial 28 pruned. Trial was pruned at epoch 2.


Best trial: 24. Best value: 0.919833: 100%|██████████| 30/30 [-35:39<00:00, -0.01it/s]

[I 2026-06-02 10:29:55,915] Trial 29 pruned. Trial was pruned at epoch 2.

Najlepsze parametry:
  num_conv_layers: 2
  activation: elu
  dropout_rate: 0.13634393255953914
  dense_units: 128
  learning_rate: 0.0006396478445223404
  filters_l0: 64
  filters_l1: 64

Najlepsza dokładność walidacyjna: 0.9198


### Testowanie najlepszego modelu

Dopasowujemy do zbioru treningowego w małej ilości epok (ponieważ nasze modele były wybierane również na małej ilości epok, więc \ zapewne najlepszy model musiał uczyć się bardzo szybko). 

In [6]:
def create_model(params):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input(shape=(28,28,1)))
    for i in range(params['num_conv_layers']):
        filters = params[f'filters_l{i}']
        activation = params['activation']
        model.add(tf.keras.layers.Conv2D(
            filters=filters, 
            kernel_size=(3, 3), 
            activation=activation, 
            padding='same'
        ))
        model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(units=params['dense_units'], activation=activation))
    model.add(tf.keras.layers.Dropout(rate=params['dropout_rate']))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=params['learning_rate']),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = create_model(study.best_params)
model.fit(X_train, y_train_h, epochs=20, batch_size=64, verbose=2)
model.evaluate(X_test, y_test_h)

Epoch 1/20
750/750 - 10s - 13ms/step - accuracy: 0.8357 - loss: 0.4599
Epoch 2/20
750/750 - 8s - 11ms/step - accuracy: 0.8870 - loss: 0.3117
Epoch 3/20
750/750 - 8s - 11ms/step - accuracy: 0.9040 - loss: 0.2666
Epoch 4/20
750/750 - 8s - 11ms/step - accuracy: 0.9119 - loss: 0.2372
Epoch 5/20
750/750 - 8s - 11ms/step - accuracy: 0.9235 - loss: 0.2098
Epoch 6/20
750/750 - 8s - 11ms/step - accuracy: 0.9281 - loss: 0.1900
Epoch 7/20
750/750 - 8s - 11ms/step - accuracy: 0.9377 - loss: 0.1695
Epoch 8/20
750/750 - 8s - 11ms/step - accuracy: 0.9438 - loss: 0.1526
Epoch 9/20
750/750 - 8s - 11ms/step - accuracy: 0.9496 - loss: 0.1338
Epoch 10/20
750/750 - 8s - 11ms/step - accuracy: 0.9567 - loss: 0.1169
Epoch 11/20
750/750 - 8s - 11ms/step - accuracy: 0.9626 - loss: 0.1024
Epoch 12/20
750/750 - 8s - 11ms/step - accuracy: 0.9658 - loss: 0.0919
Epoch 13/20
750/750 - 8s - 11ms/step - accuracy: 0.9708 - loss: 0.0797
Epoch 14/20
750/750 - 8s - 11ms/step - accuracy: 0.9738 - loss: 0.0693
Epoch 15/20
75

[0.3776080012321472, 0.920199990272522]

### Osiągnięty wynik - 92% accuracy - test set.

Jest to o 3 p. procentowe wyższy wynik w porównaniu z modelem bazowym - modelem z labu 11, \
który operuje na danych z Fashion MNIST zamiast klasycznego.